# Live Segmentation Diagnostics

Evaluate the current segmentation checkpoint on webcam/live-style frames and export a clean results table plus summary.

In [ ]:
from pathlib import Path
import sys
import json
import math

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from app.core.face_analyzer import analyze_face_image
from app.core.hair_segmentation import SEGMENTATION_CHECKPOINT_PATH, predict_hair_mask_image, predict_hair_mask_image_from_face_roi
from app.core.live_support import evaluate_supported_live_range, summarize_live_mask_quality

In [ ]:
LIVE_FRAME_DIR = PROJECT_ROOT / 'backend' / 'outputs' / 'live_segmentation_eval' / 'inputs'
OUTPUT_DIR = PROJECT_ROOT / 'backend' / 'outputs' / 'evaluations' / 'live_segmentation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}
image_paths = [path for path in sorted(LIVE_FRAME_DIR.glob('*')) if path.suffix.lower() in IMAGE_EXTS]

print('Checkpoint:', SEGMENTATION_CHECKPOINT_PATH)
print('Live frame dir:', LIVE_FRAME_DIR)
print('Frame count:', len(image_paths))
if not image_paths:
    print('Add webcam/live frames under the inputs folder and rerun.')

In [ ]:
rows = []
for image_path in image_paths:
    image = Image.open(image_path).convert('RGB')
    analysis = analyze_face_image(image_path)
    support = evaluate_supported_live_range(image, analysis)

    full_mask = predict_hair_mask_image(image)
    full_quality = summarize_live_mask_quality(analysis, full_mask)

    if analysis.face_bbox is not None:
        roi_mask = predict_hair_mask_image_from_face_roi(image, analysis.face_bbox)
        roi_quality = summarize_live_mask_quality(analysis, roi_mask)
    else:
        roi_mask = None
        roi_quality = {
            'mask_nonzero_ratio': 0.0,
            'subject_hair_bbox': None,
            'subject_hair_top_bbox': None,
            'width_ratio': None,
            'height_ratio': None,
            'center_offset_ratio': None,
            'top_offset_ratio': None,
            'mask_quality_reason': 'no_face_bbox',
            'mask_reliable': False,
        }

    rows.append({
        'filename': image_path.name,
        'face_detected': bool(analysis.face_detected),
        'landmark_count': 0 if analysis.landmarks is None else len(analysis.landmarks),
        'face_shape': analysis.face_attributes.get('face_shape', 'unknown') if analysis.face_attributes else 'unknown',
        'support_reason': support['support_reason'],
        'frame_supported': bool(support['frame_supported']),
        'yaw_proxy': support['yaw_proxy'],
        'pitch_proxy': support['pitch_proxy'],
        'roll_degrees': support['roll_degrees'],
        'face_width_ratio': support['face_width_ratio'],
        'brightness_mean': support['brightness_mean'],
        'full_mask_reason': full_quality['mask_quality_reason'],
        'full_mask_reliable': bool(full_quality['mask_reliable']),
        'full_mask_coverage': full_quality['mask_nonzero_ratio'],
        'roi_mask_reason': roi_quality['mask_quality_reason'],
        'roi_mask_reliable': bool(roi_quality['mask_reliable']),
        'roi_mask_coverage': roi_quality['mask_nonzero_ratio'],
    })

results_df = pd.DataFrame(rows)
display(results_df.head())

In [ ]:
summary = {
    'frame_count': int(len(results_df)),
    'face_detected_count': int(results_df['face_detected'].sum()) if not results_df.empty else 0,
    'supported_frame_count': int(results_df['frame_supported'].sum()) if not results_df.empty else 0,
    'support_reason_counts': {} if results_df.empty else results_df['support_reason'].value_counts().to_dict(),
    'full_mask_reason_counts': {} if results_df.empty else results_df['full_mask_reason'].value_counts().to_dict(),
    'roi_mask_reason_counts': {} if results_df.empty else results_df['roi_mask_reason'].value_counts().to_dict(),
    'full_mask_reliable_count': int(results_df['full_mask_reliable'].sum()) if not results_df.empty else 0,
    'roi_mask_reliable_count': int(results_df['roi_mask_reliable'].sum()) if not results_df.empty else 0,
}
print(json.dumps(summary, indent=2))

summary_path = OUTPUT_DIR / 'live_segmentation_summary.json'
results_path = OUTPUT_DIR / 'live_segmentation_results.csv'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
results_df.to_csv(results_path, index=False)
print('Saved summary:', summary_path)
print('Saved results:', results_path)

In [ ]:
if not results_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    results_df['support_reason'].value_counts().plot(kind='bar', ax=axes[0], title='Support reasons')
    results_df['full_mask_reason'].value_counts().plot(kind='bar', ax=axes[1], title='Full-mask reasons')
    results_df['roi_mask_reason'].value_counts().plot(kind='bar', ax=axes[2], title='ROI-mask reasons')
    plt.tight_layout()
    plt.show()